### Importing Libraries

In [ ]:
import pandas as pd
import math
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import matplotlib.pyplot as plt

### Uploading Data

In [ ]:
df1 = pd.read_csv("Consumer_Train_Dataset.csv")
df2 = pd.read_csv("Consumer_Test_Dataset.csv")

### Handling Missing Values

In [ ]:
def handle_missing_values(df):
    numerical_col = df.select_dtypes('float64').columns
    category_col = df.select_dtypes('object').columns
    category_col = [col for col in category_col if col != 'Student_ID']

    for col in numerical_col:
        df[col] = df[col].fillna(df[col].median())
    for col in category_col:
        df[col] = df[col].fillna(df[col].mode()[0])

handle_missing_values(df1)
handle_missing_values(df2)

### Getting Anslysis Ready

In [ ]:
target = 'Group'
numerical_col_1 = df1.select_dtypes('float64').columns

X = df1.drop(columns=[df1.columns[0], 'Group'], errors='ignore')
X_cat= pd.get_dummies(X, drop_first=True)

# Scaling Numerical Columns
scaler = StandardScaler()
X_num = scaler.fit_transform(df1[numerical_col_1])
X_num = pd.DataFrame(X_num, columns=numerical_col_1)

X = pd.concat([X_num, X_cat], axis=1)

# Converting A, B, C, D into mumericals
le = LabelEncoder()
Y = le.fit_transform(df1[target])

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=31)

### Linear Regression

In [ ]:
lnr = LinearRegression()
lnr.fit(X_train, y_train)

y_pred_lnr = lnr.predict(X_test)
r2_lnr = r2_score(y_test, y_pred_lnr)
rmse_lnr = math.sqrt(mean_squared_error(y_test, y_pred_lnr))
mae_lnr = mean_absolute_error(y_test, y_pred_lnr)
print(r2_lnr)
print(rmse_lnr)
print(mae_lnr)

coeff = lnr.coef_
features = X.columns

coef_df_lnr = pd.DataFrame({
    'Feature': features,
    'Coefficient': coeff
})
coef_df_lnr = coef_df_lnr.sort_values(by='Coefficient', key=abs, ascending=False)
print(coef_df_lnr)

plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred_lnr)
plt.title("Linear Regression: Predicted vs Actual")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.grid(True)
plt.show()

res_lr = y_test - y_pred_lnr

plt.figure(figsize=(6,4))
plt.hist(res_lr, bins=20)
plt.title("LR Residual Distribution")
plt.xlabel("Error")
plt.show()

### Experimenting on Different Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy:", acc)
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n")
    report = classification_report(y_test, y_pred, output_dict=True)
    print(classification_report(y_test, y_pred))
    
    # Store key metrics
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro F1": report['macro avg']['f1-score'],
        "Weighted F1": report['weighted avg']['f1-score'],
        "Macro Precision": report['macro avg']['precision'],
        "Macro Recall": report['macro avg']['recall']
    })

#Comparision between different models
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Macro F1', ascending=False)
print("\nFinal Model Comparison:\n")
print(results_df)

# For Random Forest Regressor
rf = models["Random Forest"]
importances = rf.feature_importances_
features = X.columns
rf_imp = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)
print("\nTop RF Features:\n", rf_imp.head(5))

### Final Prediction Using Logistical Regression

In [ ]:
target = 'Group'

# Separate features and target
X1 = df1.drop(columns=[df1.columns[0], target], errors='ignore')
y1 = df1[target]
X2 = df2.drop(columns=[df2.columns[0]], errors='ignore')

# Combine both datasets for consistent encoding
combined = pd.concat([X1, X2], axis=0)

# One-hot encoding
combined = pd.get_dummies(combined, drop_first=True)

# Split back
X1 = combined.iloc[:len(df1), :]
X2 = combined.iloc[len(df1):, :]

# Scale numerical columns
numerical_col = X1.select_dtypes('float64').columns

scaler = StandardScaler()
X1.loc[:, numerical_col] = scaler.fit_transform(X1[numerical_col])
X2.loc[:, numerical_col] = scaler.transform(X2[numerical_col])

# Encode target labels
le = LabelEncoder()
y1 = le.fit_transform(y1)

# Train Logistic Regression model
model = LogisticRegression(max_iter=2000)
model.fit(X1, y1)

# Predict on df2
y_pred_num = model.predict(X2)

# Convert numerical labels back
y_pred_labels = le.inverse_transform(y_pred_num)

# Store predictions
df2['Group'] = y_pred_labels

# Export final CSV
df2.to_csv("25b1242_q2.csv", index=False)